# Smart Fake News Detection — Data Analysis and Model Training

**Dataset:** 200 WELFake-labelled articles (100 fake, 100 real)  
**Models:** Logistic Regression and Multinomial Naive Bayes  
**Features:** TF-IDF word and bigram features  
**Split:** 80% training / 20% testing, stratified, random state 42

> This is a machine-learning text classifier, not an independent fact-checking system. Predictions can be wrong.

## 1. Import libraries and project utilities

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from data_utils import load_dataset
from preprocessing import clean_text

DATA_PATH = PROJECT_ROOT / "data" / "fake_news_dataset_200.csv"
MODEL_DIR = PROJECT_ROOT / "models"
VIZ_DIR = PROJECT_ROOT / "outputs" / "visualizations"
EVAL_DIR = PROJECT_ROOT / "outputs" / "evaluation_results"
for folder in (MODEL_DIR, VIZ_DIR, EVAL_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)
print("Dataset exists:", DATA_PATH.exists())

## 2. Load and inspect the dataset

In [ ]:
df = load_dataset(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
print("\\nMissing values:")
display(df.isna().sum())
print("\\nLabel counts:")
display(df["label"].value_counts())

## 3. Exploratory data analysis

In [ ]:
label_counts = df["label"].value_counts().reindex(["fake", "real"], fill_value=0)
ax = label_counts.plot(kind="bar", rot=0, title="Fake and Real News Distribution")
ax.set_xlabel("News label")
ax.set_ylabel("Number of articles")
plt.tight_layout()
plt.savefig(VIZ_DIR / "label_distribution.png", dpi=160)
plt.show()

df["word_count"] = df["combined_text"].fillna("").astype(str).str.split().str.len()
ax = df["word_count"].hist(bins=20)
ax.set_title("Article Length Distribution")
ax.set_xlabel("Words per article")
ax.set_ylabel("Frequency")
plt.tight_layout()
plt.savefig(VIZ_DIR / "article_length_distribution.png", dpi=160)
plt.show()

## 4. Clean text

In [ ]:
df["clean_text"] = df["combined_text"].map(clean_text)
df = df[df["clean_text"].str.len() > 0].copy()
print("Rows after text cleaning:", len(df))
display(df[["label", "clean_text"]].head())

## 5. Split data and create TF-IDF features

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["label"],
    test_size=0.20, random_state=42, stratify=df["label"]
)

vectorizer = TfidfVectorizer(
    max_features=50000, ngram_range=(1, 2),
    min_df=2, max_df=0.98, sublinear_tf=True
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("TF-IDF training matrix:", X_train_tfidf.shape)

## 6. Train and compare classifiers

In [ ]:
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Multinomial Naive Bayes": MultinomialNB()
}
results = {}
fitted_models = {}

for name, clf in classifiers.items():
    clf.fit(X_train_tfidf, y_train)
    pred = clf.predict(X_test_tfidf)
    fitted_models[name] = clf
    results[name] = {
        "accuracy": float(accuracy_score(y_test, pred)),
        "precision_macro": float(precision_score(y_test, pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_test, pred, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(y_test, pred, average="macro", zero_division=0)),
        "classification_report": classification_report(
            y_test, pred, output_dict=True, zero_division=0
        )
    }
    print("\\n", name)
    print(classification_report(y_test, pred, zero_division=0))

    cm = confusion_matrix(y_test, pred, labels=["fake", "real"])
    disp = ConfusionMatrixDisplay(cm, display_labels=["Fake", "Real"])
    disp.plot(values_format="d")
    disp.ax_.set_title(f"{name} Confusion Matrix")
    plt.tight_layout()
    safe_name = name.lower().replace(" ", "_")
    plt.savefig(VIZ_DIR / f"{safe_name}_confusion_matrix.png", dpi=160)
    plt.show()

summary = pd.DataFrame({
    name: {metric: scores[metric] for metric in
           ["accuracy", "precision_macro", "recall_macro", "f1_macro"]}
    for name, scores in results.items()
}).T
display(summary)

## 7. Select best model and save artifacts

In [ ]:
best_model_name = max(results, key=lambda name: results[name]["f1_macro"])
best_model = fitted_models[best_model_name]

joblib.dump(best_model, MODEL_DIR / "model.joblib")
joblib.dump(vectorizer, MODEL_DIR / "tfidf_vectorizer.joblib")

metrics = {
    "selected_model": best_model_name,
    "test_size": 0.20,
    "random_state": 42,
    "n_records_after_cleaning": int(len(df)),
    "train_records": int(len(X_train)),
    "test_records": int(len(X_test)),
    "label_mapping": {"fake": "fake", "real": "real"},
    "models": results
}
(EVAL_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")

summary.plot(kind="bar", ylim=(0, 1), rot=0, title="Classifier Comparison")
plt.ylabel("Score")
plt.tight_layout()
plt.savefig(VIZ_DIR / "model_comparison.png", dpi=160)
plt.show()

print("Selected model:", best_model_name)
print("Saved model and vectorizer in:", MODEL_DIR)
print("Saved metrics in:", EVAL_DIR)

## 8. Try a sample prediction

In [ ]:
sample_news = "Paste a news headline and article text here to test the classifier."
sample_features = vectorizer.transform([clean_text(sample_news)])
sample_prediction = best_model.predict(sample_features)[0]
print("Model prediction:", sample_prediction.upper())
if hasattr(best_model, "predict_proba"):
    print("Model scores:", dict(zip(best_model.classes_, best_model.predict_proba(sample_features)[0])))

## 9. Conclusion and limitations

- The notebook compares two supervised text classifiers using the same held-out test set.
- TF-IDF is fitted only on training data to avoid test-set vocabulary leakage.
- With only 200 articles, the test set contains just 40 examples; evaluation scores can vary substantially.
- Dataset labels and article wording may contain bias or errors.
- A prediction is not proof that a news claim is true or false. Verify important claims with reliable sources.